# Notebook 23: the ten organiser encoders that never got a result

`openadmet/chemprop-foundation-pretraining-weights` holds fifteen ChemProp encoders. nb12
embedded all fifteen, but its log only reported TabPFN results for five (`surrogate_adme`,
`whim`, `erg`, `ecfp`, `e3fp`) before it ended. Of those, only `surrogate_adme` earned a
place, and it is the one change that lifted `v25` on the board (0.5064 to 0.4938). The other
ten were never measured: `usrcat`, `rdkit2d`, `atompair`, `minimol`, `jazzy`, `pubchem`,
`osmordred`, `osmordred_pca80`, `osmordred_pca90`, `osmordred_pca95`.

**nb12 already saved every embedding to Drive** (`fnd_<name>_train.npy` and `_test.npy`, 2048
columns), so this notebook embeds nothing. It reads those files, read-only.

Two stages, so the GPU hours go to the encoders with a reason to win:

1. **Screen (CPU, a few minutes).** Each encoder is reduced to 200 principal components
   (fitted on the unlabelled train and test molecules together, no labels, as nb15 did) and
   scored with ridge regression on those components plus the qHTS and aux columns, scaffold
   5-fold, all four isoforms. `surrogate_adme` goes through the same screen as a **control**:
   if the screen cannot rank the one encoder known to work near the top, the notebook says so.
2. **TabPFN (GPU).** The top `TOP_K` encoders (default 3) get exactly the nb15 recipe: TabPFN
   2.2.1 on 200 components beside the five qHTS and four aux columns kept raw, five scaffold
   folds, a holdout fit and a blind-set fit. Set `ENCODERS` to a list to override the pick.

**Reference to beat**, nb15 (`surrogate_adme`): dev `0.5639 / 0.7268 / 0.4460 / 0.8115`,
macro **0.6370**, holdout macro **0.6713**. A new encoder does not have to beat it alone; it
has to be a different enough arm to move the blend. Test that with the `v25` blend test
afterwards, as for nb15. The screen is a proxy: ridge picked `surrogate_adme` in nb12 and
picked four encoders that TabPFN then rated poorly, so treat it as a filter and not a ranking.

**Time.** nb15 was estimated at 1 to 1.5 hours per encoder on a T4 (not measured). Three
encoders is therefore 3 to 4.5 hours. To use more GPUs, give each runtime its own `RUN` letter
and its own `ENCODERS` list.

**Safe to run beside notebooks 19 to 22, and beside copies of itself.** It reads the nb12
embeddings and the shared payload and writes only `nb23_*` files (`nb23_<encoder>_<iso>.npz`,
`part_nb23_*`, `nb23_pca_<encoder>.npz`, `nb23_screen_<RUN>.json`, `results_nb23_<RUN>.json`),
each saved under a random temporary name and renamed. The results zip holds only this run's
files. After a disconnect, Run all resumes at the next unfinished fold.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU: switch the runtime to a GPU before going on')

In [ ]:
RUN = 'a'              # a letter per runtime if you run several copies at once
ENCODERS = None        # None = the top TOP_K from the screen; or e.g. ['minimol', 'rdkit2d']
TOP_K = 3
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

CANDIDATES = ['usrcat', 'rdkit2d', 'atompair', 'minimol', 'jazzy', 'pubchem',
              'osmordred', 'osmordred_pca80', 'osmordred_pca90', 'osmordred_pca95']
CONTROL = 'surrogate_adme'     # known to work in TabPFN (nb12, nb15); screened for calibration

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/drive/MyDrive/cyp_results'
os.makedirs(OUTDIR, exist_ok=True)
assert os.path.ismount('/content/drive'), 'Drive is not mounted; the run could not resume without it'

NEED = ['meta_train.csv.gz', 'meta_test.csv.gz', 'holdout_mask.npy', 'scaffold_groups.npy',
        'qhts_tr.npy', 'qhts_te.npy', 'aux_train.npy', 'aux_test.npy']
have = sorted(os.listdir('/content/cache'))
missing = [f for f in NEED if f not in have]
assert not missing, 'payload is missing %s. Found: %s' % (missing, have)
gone = [f'fnd_{n}_{s}.npy' for n in CANDIDATES + [CONTROL] for s in ('train', 'test')
        if not os.path.exists(f'{OUTDIR}/fnd_{n}_{s}.npy')]
assert not gone, ('nb12 embeddings missing from %s: %s. They come from cyp_nb12_foundation; '
                  'rerun its embedding step.' % (OUTDIR, gone))
print('run', RUN, '| payload ok | all', len(CANDIDATES) + 1, 'embedding pairs found on Drive')

In [ ]:
!pip -q install tabpfn==2.2.1

In [ ]:
import json, os, shutil, time, uuid, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

CACHE = Path('/content/cache'); OUT = Path(OUTDIR)
ISO = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
meta = pd.read_csv(CACHE / 'meta_train.csv.gz'); mete = pd.read_csv(CACHE / 'meta_test.csv.gz')
hold = np.load(CACHE / 'holdout_mask.npy'); G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=True)
RAW = np.hstack([np.load(CACHE / 'qhts_tr.npy'), np.load(CACHE / 'aux_train.npy')]).astype(np.float32)
RAW_TE = np.hstack([np.load(CACHE / 'qhts_te.npy'), np.load(CACHE / 'aux_test.npy')]).astype(np.float32)
N, NTE = len(meta), len(mete)
assert len(RAW) == N == len(hold) == len(G) and len(RAW_TE) == NTE
Y = {i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}

def save_atomic(path, **kw):
    """Own temporary name, then rename: a half-written file is never read."""
    tmp = Path(f'{path}.{uuid.uuid4().hex}.tmp.npz')
    np.savez_compressed(tmp, **kw); os.replace(tmp, path)

def pca_features(name, k=200):
    """200 components of one encoder's frozen embedding. Fitted on train and test
    molecules together with no labels, as the nb15 payload was. Cached on Drive."""
    f = OUT / f'nb23_pca_{name}.npz'
    if f.exists():
        z = np.load(f); return z['tr'], z['te']
    a = np.load(OUT / f'fnd_{name}_train.npy').astype(np.float32)
    b = np.load(OUT / f'fnd_{name}_test.npy').astype(np.float32)
    assert a.shape[0] == N and b.shape[0] == NTE and a.shape[1] == b.shape[1], (name, a.shape, b.shape)
    X = np.vstack([a, b]); X[~np.isfinite(X)] = np.nan
    X = SimpleImputer(strategy='median').fit_transform(X)
    X = X[:, X.std(0) > 1e-8]                       # constant columns carry nothing
    X = StandardScaler().fit_transform(X)
    P = PCA(n_components=min(k, X.shape[1]), random_state=42).fit_transform(X).astype(np.float32)
    save_atomic(f, tr=P[:N], te=P[N:])
    return P[:N], P[N:]

t0 = time.time()
PCAS = {n: pca_features(n) for n in CANDIDATES + [CONTROL]}
print({n: p[0].shape[1] for n, p in PCAS.items()}, f'| {time.time()-t0:.0f}s')

## Stage 1: screen

Ridge regression on 200 components plus the nine raw qHTS and aux columns, scaffold
5-fold on the development rows, Spearman per isoform. Cheap, and it only decides which
encoders earn a TabPFN run. The control row is `surrogate_adme`: it should sit near the top.

In [ ]:
def ridge_cv(P, iso):
    y = Y[iso]; dv = np.flatnonzero(np.isfinite(y) & ~hold)
    X = np.hstack([P, RAW]); oof = np.full(len(dv), np.nan)
    for tr, va in GroupKFold(5).split(dv, dv, G[dv]):
        im = SimpleImputer(strategy='median').fit(X[dv[tr]])
        sc = StandardScaler().fit(im.transform(X[dv[tr]]))
        m = RidgeCV(alphas=np.logspace(0, 5, 11)).fit(sc.transform(im.transform(X[dv[tr]])), y[dv[tr]])
        oof[va] = m.predict(sc.transform(im.transform(X[dv[va]])))
    return float(spearmanr(y[dv], oof).statistic), oof

SCREEN_FILE = OUT / f'nb23_screen_{RUN}.json'
scores, oofs = {}, {}
for n in CANDIDATES + [CONTROL]:
    r = {i: ridge_cv(PCAS[n][0], i) for i in ISO}
    scores[n] = {i: r[i][0] for i in ISO}; scores[n]['macro'] = float(np.mean([r[i][0] for i in ISO]))
    oofs[n] = r['CYP2D6'][1]
tab = pd.DataFrame(scores).T[ISO + ['macro']].sort_values('macro', ascending=False)
print(tab.round(4).to_string())
json.dump(scores, open(SCREEN_FILE, 'w'), indent=2)

ranked = [n for n in tab.index if n != CONTROL]
rank_ctl = list(tab.index).index(CONTROL) + 1
print(f'\ncontrol {CONTROL} ranks {rank_ctl} of {len(tab)} in this screen')
if rank_ctl > len(tab) // 2:
    print('WARNING: the control is in the bottom half, so ridge does not rank these encoders the '
          'way TabPFN does. Consider setting ENCODERS by hand and running more of them.')
print('\nCYP2D6 ridge OOF, correlation between the candidates (high = near-duplicate):')
print(pd.DataFrame({n: pd.Series(oofs[n]) for n in ranked}).corr(method='spearman').round(2).to_string())

if ENCODERS is None:
    ENCODERS = ranked[:TOP_K]
bad = [n for n in ENCODERS if n not in CANDIDATES]
assert not bad, f'unknown encoders {bad}; choose from {CANDIDATES}'
print('\nTabPFN will run on:', ENCODERS)

## Stage 2: TabPFN, the nb15 recipe

Per encoder and isoform: five scaffold folds of out-of-fold predictions on the development
rows, one fit on all development rows for the holdout, one fit on development plus holdout for
the 750 blind compounds. Features are the 200 components with the nine raw qHTS and aux columns
beside them, median-imputed on the training rows only. CYP2D6 runs first, since it is the
endpoint the project is missing. Every fold is saved the moment it finishes.

In [ ]:
from tabpfn import TabPFNRegressor
KW = dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)
REF = {'CYP1A2': 0.5639, 'CYP2C9': 0.7268, 'CYP2D6': 0.4460, 'CYP3A4': 0.8115}   # nb15 dev Spearman
ORDER = ['CYP2D6', 'CYP1A2', 'CYP2C9', 'CYP3A4']

def feats(P, tr_idx, ev_P, ev_RAW):
    ri = SimpleImputer(strategy='median').fit(RAW[tr_idx])
    return (np.hstack([P[tr_idx], ri.transform(RAW[tr_idx])]).astype(np.float32),
            np.hstack([ev_P, ri.transform(ev_RAW)]).astype(np.float32))

def fit_predict(P, tr_idx, ev_P, ev_RAW, y):
    Xt, Xe = feats(P, tr_idx, ev_P, ev_RAW)
    return TabPFNRegressor(**KW).fit(Xt, y[tr_idx]).predict(Xe)

# smoke test before the long run: one small fit, shapes printed, fails loudly
P0 = PCAS[ENCODERS[0]][0]; y0 = Y['CYP3A4']; ok = np.flatnonzero(np.isfinite(y0))[:90]
Xt, Xe = feats(P0, ok[:70], P0[ok[70:]], RAW[ok[70:]])
p = TabPFNRegressor(**KW).fit(Xt, y0[ok[:70]]).predict(Xe)
print('smoke test ok: train', Xt.shape, 'eval', Xe.shape, 'finite', bool(np.isfinite(p).all()))

RESULTS = []; t_all = time.time()
for enc in ENCODERS:
    P, PT = PCAS[enc]
    for iso in ORDER:
        final = OUT / f'nb23_{enc}_{iso}.npz'
        y = Y[iso]; obs = np.isfinite(y)
        dev = np.flatnonzero(obs & ~hold); ho = np.flatnonzero(obs & hold); allr = np.flatnonzero(obs)
        if final.exists():
            z = np.load(final); oof, hp = z['oof'], z['hold']; print(f'{enc} {iso}: finished earlier, skipping', flush=True)
        else:
            part = OUT / f'part_nb23_{enc}_{iso}.npz'
            oof = np.full(len(dev), np.nan); hp = np.full(len(ho), np.nan); tp = np.full(NTE, np.nan); done = []
            if part.exists():
                z = np.load(part, allow_pickle=True); oof, hp, tp = z['oof'], z['hp'], z['tp']; done = list(z['done'])
                print(f'{enc} {iso}: resuming, finished stages {done}', flush=True)
            def ckpt(): save_atomic(part, oof=oof, hp=hp, tp=tp, done=np.array(done))
            t0 = time.time()
            for k, (tr, va) in enumerate(GroupKFold(5).split(dev, dev, G[dev])):
                if f'f{k}' in done: continue
                oof[va] = fit_predict(P, dev[tr], P[dev[va]], RAW[dev[va]], y)
                done.append(f'f{k}'); ckpt(); print(f'  {enc} {iso} fold {k} ({time.time()-t0:.0f}s)', flush=True)
            if 'hold' not in done:
                hp = fit_predict(P, dev, P[ho], RAW[ho], y); done.append('hold'); ckpt()
                print(f'  {enc} {iso} holdout model ({time.time()-t0:.0f}s)', flush=True)
            if 'test' not in done:
                tp = fit_predict(P, allr, PT, RAW_TE, y); done.append('test'); ckpt()
                print(f'  {enc} {iso} test model ({time.time()-t0:.0f}s)', flush=True)
            assert np.isfinite(oof).all() and np.isfinite(hp).all() and np.isfinite(tp).all()
            save_atomic(final, oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
            part.unlink(missing_ok=True)
        d = float(spearmanr(y[dev], oof).statistic); h = float(spearmanr(y[ho], hp).statistic)
        RESULTS.append(dict(enc=enc, iso=iso, dev_spearman=d, holdout_spearman=h, nb15=REF[iso]))
        json.dump(RESULTS, open(OUT / f'results_nb23_{RUN}.json', 'w'), indent=2)
        print(f'{enc} {iso}: dev {d:.4f} (nb15 {REF[iso]:.4f}) | holdout {h:.4f}', flush=True)

r = pd.DataFrame(RESULTS)
t = r.pivot(index='enc', columns='iso', values='dev_spearman'); t = t[[i for i in ISO if i in t.columns]]; t['macro'] = t.mean(axis=1)
h = r.pivot(index='enc', columns='iso', values='holdout_spearman'); t['holdout macro'] = h.mean(axis=1)
print('\n' + t.round(4).to_string())
print('nb15 (surrogate_adme): macro dev 0.6370 | macro holdout 0.6713')
print('total', round((time.time() - t_all) / 60, 1), 'min')

In [ ]:
# Only this run's own files, never the whole folder, which other notebooks may be writing to.
import zipfile
mine = ([OUT / f'nb23_{e}_{i}.npz' for e in ENCODERS for i in ISO]
        + [OUT / f'results_nb23_{RUN}.json', SCREEN_FILE])
with zipfile.ZipFile(f'/content/nb23_{RUN}_results.zip', 'w') as z:
    for p in mine:
        z.write(p, p.name)
try:
    from google.colab import files
    files.download(f'/content/nb23_{RUN}_results.zip')
except Exception as e:
    print('download skipped:', type(e).__name__, '- the results are already on Drive in', OUT)